In [ ]:
import os
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
import hdbscan
from hdbscan.validity import validity_index
from mlxtend.frequent_patterns import apriori, association_rules

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

sns.set_style("whitegrid")
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

### loads `.env`
#### Setting up the `database connection`

In [ ]:
load_dotenv()

pg_url = (
    f"postgresql+psycopg2://{os.getenv('PGUSER')}:{os.getenv('PGPASSWORD')}"
    f"@{os.getenv('PGHOST')}:{os.getenv('PGPORT')}/{os.getenv('PGDATABASE')}"
)

engine = create_engine(pg_url, pool_pre_ping=True)

with engine.begin() as conn:
    conn.execute(text("SET search_path TO mart, curated, public;"))

with engine.begin() as conn:
    print(conn.execute(text("SELECT now()")).scalar())

2025-12-08 10:04:10.643530-05:00


`38% (~212796)` fraudulent claims detected out of `558211` total claims
- `IP - 23402` (~11%) fraudulent records<br>
- `OP - 189394` (~89%) fraudulent records

#### 4 -  `HDBSCAN clustering` of fraudulent claims (IP & OP)

#### 4.1 Validate HDBSCAN clusters (DBCV + ARI)

In [ ]:
warnings.filterwarnings("ignore", category=RuntimeWarning)

ip_raw = pd.read_sql("SELECT * FROM mart.fraud_claims_ip", con=engine)
op_raw = pd.read_sql("SELECT * FROM mart.fraud_claims_op", con=engine)

def build_matrix_strong(df, claim_type):
    if claim_type == "IP":
        feats = ["reimb_amt", "deductible_paid", "dx_count", "px_count", "los_days",
                 "z_ip_reimb", "z_ip_los", "dup_exact_flag", "dup_near_count",
                 "overcharge_z_flag", "overcharge_iqr_flag"]
    else:
        feats = ["reimb_amt", "deductible_paid", "dx_count", "px_count",
                 "z_op_reimb", "dup_exact_flag", "dup_near_count",
                 "overcharge_z_flag", "overcharge_iqr_flag"]

    X = df[feats].apply(pd.to_numeric, errors="coerce").copy()
    X["reimb_amt_log"] = np.log1p(X["reimb_amt"].clip(lower=0))
    denom = df["reimb_amt"].replace(0, np.nan)
    X["deductible_share"] = (df["deductible_paid"] / denom).clip(0, 5)

    X = X.drop(
        columns=[c for c in ["reimb_amt", "deductible_paid"] if c in X.columns])

    nunq = X.nunique(dropna=True)
    keep_cols = nunq[nunq > 1].index.tolist()
    X = X[keep_cols]

    X = X.fillna(X.median(numeric_only=True))
    Xs = RobustScaler().fit_transform(X)

    Xs_df = pd.DataFrame(Xs, columns=keep_cols, index=df.index)
    Xs_df = Xs_df[~Xs_df.duplicated(keep="first")]
    df_clean = df.loc[Xs_df.index].copy()

    rng = np.random.RandomState(42)
    Xs_jitter = Xs_df.values + 1e-9 * rng.normal(size=Xs_df.shape)

    return df_clean, Xs_jitter, keep_cols

ip_df, Xs_ip, cols_ip = build_matrix_strong(ip_raw, "IP")
op_df, Xs_op, cols_op = build_matrix_strong(op_raw, "OP")

def pct(n, p):
    return max(10, int(np.ceil(n * p)))

def tune_hdbscan(Xs, min_cluster_size_grid, min_samples_grid,
                 methods=("leaf", "eom"), metrics=("euclidean", "manhattan")):
    best = {"dbcv": -np.inf}
    n = Xs.shape[0]
    for mcs in min_cluster_size_grid:
        for ms in min_samples_grid:
            for method in methods:
                for metric in metrics:
                    clus = hdbscan.HDBSCAN(min_cluster_size=mcs,
                                           min_samples=ms,
                                           cluster_selection_method=method,
                                           metric=metric,
                                           prediction_data=True)
                    labels = clus.fit_predict(Xs)
                    k = len(set(labels)) - (1 if -1 in labels else 0)
                    noise = float((labels == -1).mean())
                    if k >= 2:
                        try:
                            dbcv = float(validity_index(Xs, labels))
                        except Exception:
                            dbcv = -np.inf
                    else:
                        dbcv = -np.inf
                    score_tuple = (dbcv, -noise, k)
                    if score_tuple > (best["dbcv"], -best.get("noise", 1.0), best.get("k", 0)):
                        best.update({
                            "dbcv": dbcv, "noise": noise, "k": int(k),
                            "min_cluster_size": mcs, "min_samples": ms,
                            "method": method, "metric": metric,
                            "labels": labels, "model": clus
                        })
    return best

grid_ip = {"mcs": [pct(Xs_ip.shape[0], p) for p in (0.01, 0.015, 0.02)],
           "ms":  [5, 7, 10]}
grid_op = {"mcs": [pct(Xs_op.shape[0], p) for p in (0.01, 0.015, 0.02)],
           "ms":  [5, 7, 10]}

best_ip = tune_hdbscan(Xs_ip, grid_ip["mcs"], grid_ip["ms"])
best_op = tune_hdbscan(Xs_op, grid_op["mcs"], grid_op["ms"])

print("Best IP params:",
      {k: best_ip[k] for k in ["dbcv", "noise", "k", "min_cluster_size", "min_samples", "method", "metric"]})
print("Best OP params:",
      {k: best_op[k] for k in ["dbcv", "noise", "k", "min_cluster_size", "min_samples", "method", "metric"]})

ip = ip_df.copy()
ip["cluster"] = best_ip["labels"]
op = op_df.copy()
op["cluster"] = best_op["labels"]

ip_out_table = "fraud_claims_ip_clusters"
op_out_table = "fraud_claims_op_clusters"

ip.to_sql(ip_out_table, con=engine, schema="mart",
          index=False, if_exists="replace")
op.to_sql(op_out_table, con=engine, schema="mart",
          index=False, if_exists="replace")

runlog = pd.DataFrame([{
    "run_ts": datetime.utcnow().isoformat(),
    "table_in_ip": "mart.fraud_claims_ip",
    "table_in_op": "mart.fraud_claims_op",
    "table_out_ip": f"mart.{ip_out_table}",
    "table_out_op": f"mart.{op_out_table}",
    "ip_dbcv": best_ip["dbcv"], "ip_noise": best_ip["noise"], "ip_k": best_ip["k"],
    "ip_min_cluster_size": best_ip["min_cluster_size"], "ip_min_samples": best_ip["min_samples"],
    "ip_method": best_ip["method"], "ip_metric": best_ip["metric"],
    "op_dbcv": best_op["dbcv"], "op_noise": best_op["noise"], "op_k": best_op["k"],
    "op_min_cluster_size": best_op["min_cluster_size"], "op_min_samples": best_op["min_samples"],
    "op_method": best_op["method"], "op_metric": best_op["metric"]
}])

print("\nWrote:")
print(f"  mart.{ip_out_table}  (rows={len(ip)})")
print(f"  mart.{op_out_table}  (rows={len(op)})")
print("  mart.hdbscan_runs    (+1 row)")

c:\Users\zayed\anaconda3\envs\KNCVU\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\zayed\anaconda3\envs\KNCVU\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\zayed\anaconda3\envs\KNCVU\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\zayed\anaconda3\envs\KNCVU\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\zayed\anaconda3\envs\KNCVU\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 

c:\Users\zayed\anaconda3\envs\KNCVU\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\zayed\anaconda3\envs\KNCVU\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\zayed\anaconda3\envs\KNCVU\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\zayed\anaconda3\envs\KNCVU\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\zayed\anaconda3\envs\KNCVU\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 

Best IP params: {'dbcv': 0.1296742408195641, 'noise': 0.31559187279151946, 'k': 19, 'min_cluster_size': 340, 'min_samples': 10, 'method': 'leaf', 'metric': 'manhattan'}
Best OP params: {'dbcv': 0.18808329243753363, 'noise': 0.155209324452902, 'k': 11, 'min_cluster_size': 127, 'min_samples': 7, 'method': 'eom', 'metric': 'manhattan'}


c:\Users\zayed\anaconda3\envs\KNCVU\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\zayed\anaconda3\envs\KNCVU\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\zayed\anaconda3\envs\KNCVU\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\zayed\anaconda3\envs\KNCVU\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\zayed\anaconda3\envs\KNCVU\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 

Best IP params: {'dbcv': 0.1296742408195641, 'noise': 0.31559187279151946, 'k': 19, 'min_cluster_size': 340, 'min_samples': 10, 'method': 'leaf', 'metric': 'manhattan'}
Best OP params: {'dbcv': 0.18808329243753363, 'noise': 0.155209324452902, 'k': 11, 'min_cluster_size': 127, 'min_samples': 7, 'method': 'eom', 'metric': 'manhattan'}


InternalError: (psycopg2.errors.DependentObjectsStillExist) cannot drop table fraud_claims_ip_clusters because other objects depend on it
DETAIL:  view cluster_signature_hits_fraudonly_enriched depends on table fraud_claims_ip_clusters
view claim_signature_alerts_fraudonly depends on view cluster_signature_hits_fraudonly_enriched
HINT:  Use DROP ... CASCADE to drop the dependent objects too.

[SQL: 
DROP TABLE mart.fraud_claims_ip_clusters]
(Background on this error at: https://sqlalche.me/e/20/2j85)

#### 4.2 — Cluster cards (fraudulent IP & OP claims)

In [ ]:
ip = pd.read_sql("SELECT * FROM mart.fraud_claims_ip_clusters", con=engine)
op = pd.read_sql("SELECT * FROM mart.fraud_claims_op_clusters", con=engine)

def _present(cols, df):
    return [c for c in cols if c in df.columns]


def make_cluster_cards(df, claim_type):
    core = df[df["cluster"] != -1].copy()
    if core.empty:
        return pd.DataFrame()

    money = core.groupby("cluster")["reimb_amt"].agg(
        n="count", total_paid="sum", median_paid="median"
    )

    base_feats = ["dx_count", "px_count", "dup_exact_flag", "dup_near_count",
                  "overcharge_z_flag", "overcharge_iqr_flag"]
    if claim_type == "IP":
        base_feats += ["los_days", "z_ip_reimb", "z_ip_los"]
    else:
        base_feats += ["z_op_reimb"]

    feat_cols = _present(base_feats, core)
    stats = core.groupby("cluster")[feat_cols].median(numeric_only=True)
    cards = (money.join(stats)
                  .reset_index()
                  .sort_values(["n", "total_paid"], ascending=[False, False]))
    return cards


ip_cards = make_cluster_cards(ip, "IP")
op_cards = make_cluster_cards(op, "OP")

ip_cards.to_csv("fraud_ip_cluster_cards.csv", index=False)
op_cards.to_csv("fraud_op_cluster_cards.csv", index=False)
print("Saved cluster cards -> fraud_ip_cluster_cards.csv, fraud_op_cluster_cards.csv")

def build_matrix_for_pca(df, claim_type):
    if claim_type == "IP":
        feats = ["reimb_amt", "deductible_paid", "dx_count", "px_count", "los_days",
                 "z_ip_reimb", "z_ip_los", "dup_exact_flag", "dup_near_count",
                 "overcharge_z_flag", "overcharge_iqr_flag"]
    else:
        feats = ["reimb_amt", "deductible_paid", "dx_count", "px_count",
                 "z_op_reimb", "dup_exact_flag", "dup_near_count",
                 "overcharge_z_flag", "overcharge_iqr_flag"]

    X = df[_present(feats, df)].apply(pd.to_numeric, errors="coerce").copy()

    if "reimb_amt" in X.columns:
        X["reimb_amt_log"] = np.log1p(X["reimb_amt"].clip(lower=0))
    if "deductible_paid" in X.columns and "reimb_amt" in df.columns:
        denom = df["reimb_amt"].replace(0, np.nan)
        X["deductible_share"] = (df["deductible_paid"] / denom).clip(0, 5)

    X = X.drop(
        columns=[c for c in ["reimb_amt", "deductible_paid"] if c in X.columns])

    nunq = X.nunique(dropna=True)
    keep = nunq[nunq > 1].index.tolist()
    if not keep:
        raise ValueError(
            "All features constant/empty after filtering; cannot plot PCA.")
    X = X[keep].fillna(X.median(numeric_only=True))

    Xs = RobustScaler().fit_transform(X)
    return Xs


def pca_2d(Xs):
    pca = PCA(n_components=2, random_state=42)
    Z = pca.fit_transform(Xs)
    return Z, pca.explained_variance_ratio_.sum()


def plot_pca_scatter(Z, labels, title, fname):
    """Professional-quality PCA scatter plot with seaborn styling"""
    labels = np.asarray(labels)
    uniq = np.unique(labels)
    
    plot_df = pd.DataFrame({
        'PCA1': Z[:, 0],
        'PCA2': Z[:, 1],
        'Cluster': labels.astype(str)
    })
    
    noise_df = plot_df[plot_df['Cluster'] == '-1']
    cluster_df = plot_df[plot_df['Cluster'] != '-1']
    
    sns.set_context("paper", font_scale=1.3)
    sns.set_style("whitegrid", {
        'grid.linestyle': '--',
        'grid.alpha': 0.3,
        'axes.edgecolor': '.2',
        'axes.linewidth': 1.2
    })
    
    fig, ax = plt.subplots(figsize=(14, 9), dpi=150)
    
    if len(noise_df) > 0:
        ax.scatter(noise_df['PCA1'], noise_df['PCA2'], 
                  s=20, alpha=0.2, color='#d3d3d3', 
                  label='Noise (-1)', edgecolors='none', rasterized=True)
    
    if len(cluster_df) > 0:
        unique_clusters = sorted(cluster_df['Cluster'].unique(), key=lambda x: int(x))
        n_clusters = len(unique_clusters)
        
        if n_clusters <= 10:
            palette = sns.color_palette("tab10", n_clusters)
        else:
            palette = sns.color_palette("husl", n_clusters)
        
        for i, cluster in enumerate(unique_clusters):
            cluster_points = cluster_df[cluster_df['Cluster'] == cluster]
            ax.scatter(cluster_points['PCA1'], cluster_points['PCA2'],
                      s=45, alpha=0.75, color=palette[i],
                      label=f'Cluster {cluster}', 
                      edgecolors='white', linewidth=0.6)
    
    ax.set_title(title, fontsize=16, fontweight='bold', pad=20, family='sans-serif')
    ax.set_xlabel('PCA Component 1', fontsize=13, fontweight='bold', family='sans-serif')
    ax.set_ylabel('PCA Component 2', fontsize=13, fontweight='bold', family='sans-serif')
    
    ax.grid(True, alpha=0.25, linestyle='--', linewidth=0.8, color='gray')
    ax.set_facecolor('#fafafa')
    
    if len(cluster_df) > 0:
        ncol = 1 if n_clusters <= 8 else 2
        legend = ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5),
                          frameon=True, shadow=True, fancybox=True,
                          fontsize=10, ncol=ncol, markerscale=1.2)
        legend.get_frame().set_facecolor('white')
        legend.get_frame().set_alpha(0.95)
        legend.get_frame().set_edgecolor('gray')
        legend.get_frame().set_linewidth(1)
    
    plt.tight_layout()
    
    plt.savefig(fname, bbox_inches="tight", dpi=300, facecolor='white')
    plt.show()
    
    sns.set_style("whitegrid")


Xs_ip = build_matrix_for_pca(ip, "IP")
Z_ip, ip_var = pca_2d(Xs_ip)
plot_pca_scatter(Z_ip, ip["cluster"].to_numpy(),
                 "Fraudulent IP Claims — HDBSCAN Clustering (PCA Projection)",
                 "fraudulent_ip_claims_pca.png")

Xs_op = build_matrix_for_pca(op, "OP")
Z_op, op_var = pca_2d(Xs_op)
plot_pca_scatter(Z_op, op["cluster"].to_numpy(),
                 "Fraudulent OP Claims — HDBSCAN Clustering (PCA Projection)",
                 "fraudulent_op_claims_pca.png")

print(f"IP PCA variance explained (2D): {ip_var:.3f}")
print(f"OP PCA variance explained (2D): {op_var:.3f}")

NameError: name 'pd' is not defined

#### 4.3 - mine association rules (dx/px patterns) inside each HDBSCAN cluster

In [ ]:
ip = pd.read_sql(
    "SELECT claimid, cluster FROM mart.fraud_claims_ip_clusters", con=engine)
op = pd.read_sql(
    "SELECT claimid, cluster FROM mart.fraud_claims_op_clusters", con=engine)
codes = pd.read_sql(
    "SELECT claimid, item FROM mart.claim_codes", con=engine).drop_duplicates()

def mine_rules_by_cluster(df_claims, claim_type, min_support=0.02, min_conf=0.6, min_lift=1.2, top_items=300):
    cc = df_claims[df_claims["cluster"] != -
                   1].merge(codes, on="claimid", how="inner")
    results = []

    for c in sorted(cc["cluster"].unique()):
        sub = cc[cc["cluster"] == c][["claimid", "item"]].drop_duplicates()

        popular = sub["item"].value_counts().head(top_items).index
        sub = sub[sub["item"].isin(popular)]

        basket = pd.crosstab(sub["claimid"], sub["item"]).astype(bool)

        ms = max(min_support, 30 / max(1, basket.shape[0]))

        itemsets = apriori(basket, min_support=ms, use_colnames=True)
        if itemsets.empty:
            continue

        rules = association_rules(
            itemsets, metric="confidence", min_threshold=min_conf)
        if rules.empty:
            continue

        rules = rules[rules["lift"] >= min_lift].copy()
        if rules.empty:
            continue

        n_claims = basket.shape[0]
        rules["cluster"] = c
        rules["claim_type"] = claim_type
        rules["n_in_cluster"] = n_claims
        rules["antecedents_str"] = rules["antecedents"].apply(
            lambda s: "+".join(sorted(map(str, s))))
        rules["consequents_str"] = rules["consequents"].apply(
            lambda s: "+".join(sorted(map(str, s))))
        rules["antecedent_cov"] = (
            rules["support"] * n_claims).round().astype(int)

        keep = ["claim_type", "cluster", "n_in_cluster", "antecedents_str", "consequents_str",
                "support", "confidence", "lift", "antecedent_cov"]
        results.append(rules[keep].sort_values(
            ["lift", "support"], ascending=False))

    if not results:
        return pd.DataFrame(columns=["claim_type", "cluster", "n_in_cluster", "antecedents_str", "consequents_str",
                                     "support", "confidence", "lift", "antecedent_cov"])
    return pd.concat(results, ignore_index=True)


ip_rules = mine_rules_by_cluster(
    ip, "IP", min_support=0.02, min_conf=0.6, min_lift=1.2, top_items=300)
op_rules = mine_rules_by_cluster(
    op, "OP", min_support=0.02, min_conf=0.6, min_lift=1.2, top_items=300)

ip_rules.to_csv("fraud_ip_association_rules.csv", index=False)
op_rules.to_csv("fraud_op_association_rules.csv", index=False)
both = pd.concat([ip_rules, op_rules], ignore_index=True)
both.to_csv("fraud_association_rules_all.csv", index=False)

print("Saved:",
      "fraud_ip_association_rules.csv,",
      "fraud_op_association_rules.csv,",
      "fraud_association_rules_all.csv")

display_cols = ["claim_type", "cluster", "n_in_cluster", "antecedents_str", "consequents_str",
                "support", "confidence", "lift", "antecedent_cov"]
print("\nTop rules (by lift):")
print(both.sort_values(["lift", "support"], ascending=False)[
      display_cols].head(10).to_string(index=False))

Saved: fraud_ip_association_rules.csv, fraud_op_association_rules.csv, fraud_association_rules_all.csv

Top rules (by lift):
claim_type  cluster  n_in_cluster antecedents_str consequents_str  support  confidence  lift  antecedent_cov
        OP        1          2419         280+285             588     0.04        0.77 10.13              95
        OP        1          2419         285+588             280     0.04        0.69  9.46              95
        IP       18          1573             995             038     0.03        0.64  8.90              49
        OP        1          2419             V56             588     0.02        0.64  8.43              58
        OP        1          2419             588             280     0.05        0.61  8.41             112
        OP        1          2419             280             588     0.05        0.64  8.41             112
        IP       14          4107            8154             715     0.03        0.86  7.42             126
   